# LangGraph: Learn, Operate, and Design for Production

A single practical guide for this repository. It covers graph basics, runnable no-model examples, agent patterns, and the production safeguards needed when an agent can invoke tools.

**Audience:** Python developers learning how LangGraph fits into the SERA agent runtime.

**Important:** an LLM can decide *which action to request*. Application code must still validate commands, authorize tools, enforce budgets, persist state, and record audit events.

## Visual reference: agent patterns

![LangGraph agent patterns](../app/agent/docs/assets/langgraph-agent-patterns.png)

Start with one tool-using agent. Use subagents, routing, or handoffs only when different domains, context boundaries, direct user conversations, or parallel work justify the added complexity.

## 1. Install and run

This project has LangChain providers but does not list `langgraph` as a direct dependency. Add it when you are ready to use these examples:

```bash
pip install -U langgraph
# or, if your environment has uv
uv add langgraph
```

Activate the virtual environment first, then run the cells below. The first two examples need no API key or model.

## 2. Core model

LangGraph executes a stateful graph:

```text
state → node → edge → next node → updated state
```

- **State:** the explicit data shared between steps.
- **Node:** a Python function, model call, tool executor, validator, or subgraph.
- **Edge:** a fixed transition or routing decision.
- **Checkpointer:** optional durable storage for graph snapshots and resumable threads.

Nodes should return state updates rather than mutate the incoming state. Keep policy, authorization, schemas, and budget enforcement deterministic and outside the model.

In [ ]:
from typing import TypedDict
from langgraph.graph import START, END, StateGraph







In [ ]:
from typing_extensions import TypedDict
from langgraph.graph import END, START, StateGraph


class LessonState(TypedDict):
    topic: str
    lesson: str


def make_lesson(state: LessonState) -> dict[str, str]:
    return {"lesson": f"LangGraph represents {state['topic']} as a graph."}


builder = StateGraph(LessonState)
builder.add_node("make_lesson", make_lesson)
builder.add_edge(START, "make_lesson")
builder.add_edge("make_lesson", END)
basic_graph = builder.compile()

basic_graph.invoke({"topic": "agent workflows"})

## 3. Conditional routing

A conditional edge uses the current state to select the next named node. In production, use deterministic routing for known business rules. An LLM may classify ambiguous requests, but validate the resulting structured decision before following it.

In [ ]:
from typing import Literal
from typing_extensions import TypedDict
from langgraph.graph import END, START, StateGraph


class RouteState(TypedDict):
    request: str
    route: str
    response: str


def classify(state: RouteState) -> dict[str, str]:
    route = "documentation" if "docs" in state["request"].lower() else "general"
    return {"route": route}


def choose_next(state: RouteState) -> Literal["documentation", "general"]:
    return state["route"]  # The classifier returns one of the declared routes.


def answer_docs(_: RouteState) -> dict[str, str]:
    return {"response": "Open the documentation learning path first."}


def answer_general(_: RouteState) -> dict[str, str]:
    return {"response": "Start with the basic graph example."}


builder = StateGraph(RouteState)
builder.add_node("classify", classify)
builder.add_node("documentation", answer_docs)
builder.add_node("general", answer_general)
builder.add_edge(START, "classify")
builder.add_conditional_edges("classify", choose_next)
builder.add_edge("documentation", END)
builder.add_edge("general", END)
routing_graph = builder.compile()

routing_graph.invoke({"request": "Where are the docs?"})

## 4. Agent patterns: when to use each

| Pattern | Best use | Important trade-off |
| --- | --- | --- |
| Tool-using agent | A manageable set of tools | Simplest and easiest to trace |
| Supervisor + subagents | Central agent delegates isolated specialist work | More calls, but clean context separation |
| Router / fan-out | Clear domains that can run in parallel | Needs synthesis and explicit result reducers |
| Handoffs | A specialist must converse directly with the user | Persist active state and preserve valid messages |
| Custom workflow | Policy, approvals, loops, or business steps own part of the path | Most control; requires deliberate graph design |

Do not make a system multi-agent merely because the task is complex. A single agent with focused tools is usually the right first implementation.

## 5. Production runtime architecture

![Production LangGraph agent runtime](../app/agent/docs/assets/production-langgraph-agent-runtime.png)

A production run should follow this ownership model:

1. **Client/API:** receives an idempotent command; it does not directly mutate graph state.
2. **Command validation:** validates identity, workspace scope, input schema, and protocol.
3. **LangGraph orchestrator:** coordinates named nodes and resumes the model/tool loop.
4. **Policy + approval:** decides allow, ask, or deny from deterministic policy—not model preference.
5. **Tool executor:** validates the typed tool contract, enforces limits, executes, and records exactly one result.
6. **Checkpoint store:** persists each graph step so interruptions, approvals, and failures can resume safely.
7. **Event stream + observability:** makes visible progress, audit evidence, metrics, and traces available to clients and operators.

## 6. How to operate safely

Use `invoke()` for a completed result and `stream()` for visible progress. For multi-turn work, pass a stable thread identifier and configure a persistent checkpointer.

```python
result = graph.invoke(input_state)
for event in graph.stream(input_state):
    publish_visible_event(event)
```

Before connecting a real model or tool, enforce these rules:

- Use typed state and typed tool arguments.
- Run tools only through SERA's central executor and permission engine.
- Apply model-call, tool-call, cost, token, deadline, cancellation, recursion, and no-progress guards.
- Persist checkpoints before pauses and use a durable decision record for approvals.
- Pair every model tool call with exactly one tool result before the next model call.
- Emit ordered visible events; never reconstruct audit history from application logs.
- Keep secrets in environment variables or a secret manager, never in notebooks or committed source.

## 7. SERA integration boundary

The implementation belongs under `app/agent/`. Keep FastAPI routes as adapters that validate a command and start or resume a run. Keep tool policy and execution in their own layers.

The relevant repository source is [`app/agent/contracts.py`](../app/agent/contracts.py). It defines tool capability contracts, risk classes, permissions, idempotency, cancellation behavior, and result states. The detailed target implementation is documented in [`app/agent/docs/agent-architecture`](../app/agent/docs/agent-architecture/README.md).

## 8. Official production resources and diagrams

- [LangGraph overview](https://docs.langchain.com/oss/python/langgraph/overview) — durable execution, memory, human review, debugging, and deployment.
- [Graph API](https://docs.langchain.com/oss/python/langgraph/graph-api) — state, nodes, and fixed/conditional edges.
- [Workflows and agents](https://docs.langchain.com/oss/python/langgraph/workflows-agents) — official diagrams for prompt chaining, routing, parallelization, orchestrator-worker, evaluator-optimizer, and agents.
- [Persistence](https://docs.langchain.com/oss/python/langgraph/persistence) — official checkpoint visual; use persistent stores for production.
- [Human-in-the-loop](https://docs.langchain.com/oss/python/langchain/human-in-the-loop) — pause, approve, edit, reject, and resume tool actions.
- [Multi-agent patterns](https://docs.langchain.com/oss/python/langchain/multi-agent/index) — official visuals for subagents, handoffs, skills, and routers.
- [SERA agent architecture](../app/agent/docs/agent-architecture/README.md) — the repository's production target and invariants.